In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data = datasets.FashionMNIST(
    root = "data",
    train = True,
    download = True,
    transform = ToTensor()
)

test_data = datasets.FashionMNIST(
    root = "data",
    train = False,
    download = True,
    transform = ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size = 64)
test_dataloader = DataLoader(test_data, batch_size = 64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    
model = NeuralNetwork()


# Hyperparameters

In [3]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

# Loss Function

In [4]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()  # nn.CrossEntropyLoss combines nn.LogSoftmax and nn.NLLLoss.

# Optimizer

In [5]:
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

Inside the training loop, optimization happens in three steps:

1. Call optimizer.zero_grad() to reset the gradients of model parameters. Gradients by default add up; to prevent double-counting, we explicitly zero them at each iteration.

2. Backpropagate the prediction loss with a call to loss.backward(). PyTorch deposits the gradients of the loss w.r.t. each parameter.

3. Once we have our gradients, we call optimizer.step() to adjust the parameters by the gradients collected in the backward pass.

# Full Implementation
We define train_loop that loops over our optimization code, and test_loop that evaluates the model’s performance against our test data.

In [6]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: /n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")



In [7]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 20
for t in range(epochs):
    print(f"Epoch {t+1}\n--------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")


Epoch 1
--------------------------
loss: 2.307939 [   64/60000]
loss: 2.292613 [ 6464/60000]
loss: 2.277694 [12864/60000]
loss: 2.270406 [19264/60000]
loss: 2.250165 [25664/60000]
loss: 2.224329 [32064/60000]
loss: 2.226256 [38464/60000]
loss: 2.195102 [44864/60000]
loss: 2.186261 [51264/60000]
loss: 2.157733 [57664/60000]
Test Error: /n Accuracy: 46.6%, Avg loss: 2.155481 

Epoch 2
--------------------------
loss: 2.167840 [   64/60000]
loss: 2.159214 [ 6464/60000]
loss: 2.106899 [12864/60000]
loss: 2.118329 [19264/60000]
loss: 2.069111 [25664/60000]
loss: 2.009231 [32064/60000]
loss: 2.033764 [38464/60000]
loss: 1.956292 [44864/60000]
loss: 1.955949 [51264/60000]
loss: 1.888150 [57664/60000]
Test Error: /n Accuracy: 54.6%, Avg loss: 1.891701 

Epoch 3
--------------------------
loss: 1.929080 [   64/60000]
loss: 1.900896 [ 6464/60000]
loss: 1.786710 [12864/60000]
loss: 1.821336 [19264/60000]
loss: 1.713450 [25664/60000]
loss: 1.665898 [32064/60000]
loss: 1.681609 [38464/60000]
loss: 